# 04 · Silver — Cotizaciones

Unifica las cotizaciones de todas las fuentes en `silver.cotizaciones`, con precios comparables a los operados.

```
bronze.data912_cotizaciones ─ conform_data912: renta fija ÷ 100 (cotiza cada 100 VN) ─┐
                                                                                        ├─▶ silver.cotizaciones (simbolo, fecha, fuente)
bronze.yahoo_cotizaciones   ─ conform_yahoo: revierte splits posteriores ──────────────┘   es_preferida = menor prioridad del registro
```

| Regla | Criterio | Decisión |
|---|---|---|
| Última extracción | Por (símbolo, fecha) se usa la extracción más reciente; si un símbolo falló en la última, se conserva el dato anterior | DT-021 |
| Escala | data912 publica la renta fija cada 100 VN; el dataset, por 1 VN | DT-021 |
| Splits | Yahoo ajusta retroactivamente: precio real = publicado × splits posteriores | DT-017 |
| Fuente preferida | Por (símbolo, fecha), la de menor prioridad para el tipo de instrumento; las demás se conservan | DT-019 |

**Prerrequisito:** `setup/06_cotizaciones.sql` y el notebook `03_cotizaciones_ingesta`.

## Parámetros

In [0]:
dbutils.widgets.text("catalog", "workspace", "Catálogo")
CATALOG = dbutils.widgets.get("catalog")

In [0]:
import json
import os
import sys
from datetime import datetime, timezone

# La lógica vive en src/ para poder testearla sin cluster (ver tests/).
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

from pyspark.sql import functions as F

from iol_pipeline.config import PipelineConfig
from iol_pipeline.silver_cotizaciones import build_silver_quotes
from iol_pipeline.sources import QUOTE_SOURCES

spark.conf.set("spark.sql.session.timeZone", "UTC")

cfg = PipelineConfig(catalog=CATALOG)
silver_table = cfg.silver_table("transacciones")
quotes_table = cfg.silver_table("cotizaciones")
bronze_tables = {s.name: cfg.bronze_table(s.bronze_table) for s in QUOTE_SOURCES}

for name, table in bronze_tables.items():
    print(f"{name:8} {table}")
print(f"destino  {quotes_table}")

data912  workspace.bronze.data912_cotizaciones
yahoo    workspace.bronze.yahoo_cotizaciones
destino  workspace.silver.cotizaciones


## Precondiciones

In [0]:
for table in [silver_table, quotes_table, *bronze_tables.values()]:
    assert spark.catalog.tableExists(table), f"No existe {table}. Ejecutar setup/06_cotizaciones.sql"

bronze_rows = {name: spark.table(t).count() for name, t in bronze_tables.items()}
print("Filas en bronze por fuente:", bronze_rows)
assert sum(bronze_rows.values()) > 0, "No hay cotizaciones en bronze. Ejecutar 03_cotizaciones_ingesta"

Filas en bronze por fuente: {'data912': 52131, 'yahoo': 114742}


## Normalización y fuente preferida

In [0]:
silver = spark.table(silver_table)
instrument_types = silver.select("simbolo", "tipo_instrumento").distinct()
processed_at = datetime.now(timezone.utc).replace(tzinfo=None)

build_silver_quotes(spark, bronze_tables, instrument_types, QUOTE_SOURCES, processed_at).writeTo(quotes_table).overwrite(F.lit(True))
quotes = spark.table(quotes_table)

display(
    quotes.groupBy("fuente")
    .agg(F.count("*").alias("filas"), F.sum(F.col("es_preferida").cast("int")).alias("preferidas"),
         F.countDistinct("simbolo").alias("simbolos"), F.min("fecha").alias("desde"), F.max("fecha").alias("hasta"))
    .orderBy("fuente")
)

fuente,filas,preferidas,simbolos,desde,hasta
data912,52131,52131,261,2025-12-18,2026-10-15
yahoo,114719,75483,672,2025-12-18,2026-10-16


## Reversión de splits (Yahoo)
Símbolos con splits posteriores al período: sus precios publicados se multiplican por el factor para volver a la escala que tenían ese día.

In [0]:
display(
    quotes.where("fuente = 'yahoo' AND factor_ajuste <> 1")
    .groupBy("simbolo").agg(F.max("factor_ajuste").alias("factor_maximo"), F.max("fecha").alias("ultimo_dia_ajustado"))
    .orderBy("simbolo")
)

simbolo,factor_maximo,ultimo_dia_ajustado
BKNG,25.0,2026-04-01
DD,0.333333,2026-06-23
ETHA,0.333333,2026-10-05
ETHAD,0.333333,2026-10-05
HUT,25.0,2026-05-28
HUTD,25.0,2026-05-28
ITUB,1.03,2025-12-26
ITUBD,1.03,2025-12-26
MIRG,10.0,2026-07-31
SCCO,1.030808,2026-08-10


## Consistencia entre fuentes
Donde ambas fuentes cotizan el mismo símbolo y día, sus cierres normalizados deberían coincidir. Es la validación independiente de la reversión de splits y de la escala.

In [0]:
pairs = (
    quotes.where("fuente = 'data912'").select("simbolo", "fecha", F.col("cierre").alias("cierre_data912"))
    .join(quotes.where("fuente = 'yahoo'").select("simbolo", "fecha", F.col("cierre").alias("cierre_yahoo"), "factor_ajuste"),
          ["simbolo", "fecha"])
    .withColumn("dif_pct", F.abs(F.col("cierre_yahoo") / F.col("cierre_data912") - 1) * 100)
)
display(pairs.agg(
    F.countDistinct("simbolo").alias("simbolos_en_ambas"),
    F.count("*").alias("dias_comparados"),
    F.round(F.expr("percentile(dif_pct, 0.5)"), 3).alias("dif_mediana_pct"),
    F.round(F.expr("percentile(dif_pct, 0.9)"), 3).alias("dif_p90_pct"),
    F.sum((F.col("dif_pct") > 10).cast("int")).alias("dias_con_dif_mayor_10pct"),
))

simbolos_en_ambas,dias_comparados,dif_mediana_pct,dif_p90_pct,dias_con_dif_mayor_10pct
196,39236,0.091,0.635,559


Ejemplo: símbolos con split, el mismo día en ambas fuentes.

In [0]:
display(
    pairs.where("factor_ajuste <> 1").orderBy("simbolo", "fecha")
    .groupBy("simbolo").agg(F.first("fecha").alias("fecha"), F.first("factor_ajuste").alias("factor_split"),
                            F.first("cierre_yahoo").alias("cierre_yahoo_revertido"), F.first("cierre_data912").alias("cierre_data912"))
)

simbolo,fecha,factor_split,cierre_yahoo_revertido,cierre_data912
DD,2025-12-18,0.333333,12529.98747,12530.0
HUT,2025-12-18,25.0,298575.0,299475.0
ITUB,2025-12-18,1.03,10980.000166015625,10930.0
MIRG,2025-12-18,10.0,276000.0,27700.0
SPY,2025-12-18,3.0,52300.001953125,52325.0
YPFD,2025-12-18,10.0,54750.0,54800.0


## Escala contra lo operado
Mediana de `precio operado / cierre preferido` por fuente y tipo de instrumento, en el mismo símbolo y fecha operativa. Debe dar cerca de 1: confirma que precio operado y cierre están en la misma unidad.

In [0]:
preferred = quotes.where("es_preferida").select("simbolo", F.col("fecha").alias("fecha_operativa"), "fuente", "cierre")
display(
    silver.where("dq_status = 'OK'").join(preferred, ["simbolo", "fecha_operativa"])
    .withColumn("ratio", F.col("precio").cast("double") / F.col("cierre"))
    .groupBy("fuente", "tipo_instrumento")
    .agg(F.count("*").alias("operaciones"), F.round(F.expr("percentile(ratio, 0.5)"), 4).alias("ratio_mediana"),
         F.round(F.expr("percentile(ratio, 0.1)"), 4).alias("ratio_p10"), F.round(F.expr("percentile(ratio, 0.9)"), 4).alias("ratio_p90"))
    .orderBy("fuente", F.desc("operaciones"))
)

fuente,tipo_instrumento,operaciones,ratio_mediana,ratio_p10,ratio_p90
data912,CEDEAR,27011,1.0009,0.9786,1.0275
data912,ACCION,15310,1.0015,0.9656,1.0401
data912,BONO,14742,0.9998,0.9912,1.0097
data912,ON,45,1.0043,0.9826,1.0219
yahoo,CEDEAR,18232,1.0011,0.972,1.0383
yahoo,ACCION,1102,1.0028,0.9667,1.0468


## Resultado

In [0]:
rows = quotes.count()
summary = {
    "table": quotes_table,
    "rows": rows,
    "status": "OK" if rows > 0 else "FAIL",
    "message": f"{rows} cotizaciones normalizadas de {len(QUOTE_SOURCES)} fuentes" if rows else "No se generaron cotizaciones",
}
print(json.dumps(summary, indent=2))
dbutils.notebook.exit(json.dumps(summary))